# 🧬 End-to-End: DNABERT-2 Encoder → mCNN Classifier

## Architecture
```
DNA Sequence (101bp string)
       │
       ▼
┌──────────────────────────┐
│   BPE Tokenizer          │  (on-the-fly, per batch)
│   → input_ids (105,)     │
│   → attention_mask       │
└──────────────────────────┘
       │
       ▼
┌──────────────────────────┐
│   DNABERT-2 (117M)       │  FROZEN encoder
│   → (batch, 105, 768)    │  contextualized embeddings
└──────────────────────────┘
       │
       ▼
┌──────────────────────────┐
│   Multi-Scale CNN        │  TRAINABLE classifier head
│   Conv1D k=[1,2,3]       │  (Optimized to prevent overfitting)
│   → GlobalMaxPool        │
│   → FC → 4 classes       │
└──────────────────────────┘
       │
       ▼
  SP1 / SP2 / SP4 / Negative
```

### Why End-to-End?
- **No intermediate files**: No need to extract and save 6GB `.npy` embedding files.
- **Low RAM**: Only the current batch exists in memory at any time (~40MB per batch).
- **Clean pipeline**: Single model, single forward pass, from raw string to prediction.
- **Fine-tuning ready**: Can unfreeze DNABERT-2 later for joint fine-tuning.

### Cell 1: Setup

In [ ]:
!pip install -q transformers einops safetensors huggingface_hub scikit-learn matplotlib seaborn

import os, subprocess, sys

REPO_URL = "https://github.com/JustinYuanZe/SP1_TF_Biding_Project.git"
REPO_NAME = "SP1_TF_Biding_Project"

if os.path.basename(os.getcwd()) == REPO_NAME:
    os.chdir("..")

if not os.path.isdir(REPO_NAME):
    print("Cloning...")
    subprocess.run(["git", "clone", REPO_URL], check=True)
else:
    print("Pulling latest...")
    subprocess.run(["git", "-C", REPO_NAME, "pull"], check=True)

os.chdir(REPO_NAME)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print(f"Ready: {os.getcwd()}")

### Cell 2: Load Data + DNABERT-2

In [ ]:
import torch
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

from src.dnabert_wrapper import DNABERTWrapper
from src.mcnn_model import MultiScaleCNN
from src.pipeline_utils import DNAPipelineDataset, DNABERT_mCNN
from src.train import train_model, evaluate_model, plot_curves

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# --- Load FASTA ---
def load_fasta(path):
    seqs = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line.startswith('>'):
                seqs.append(line.upper())
    return seqs

print("Loading datasets...")
seqs_sp1 = load_fasta("data/processed/sp1_positive_final.fasta")
seqs_sp2 = load_fasta("data/processed/sp2_positive_final.fasta")
seqs_sp4 = load_fasta("data/processed/sp4_positive_final.fasta")
seqs_neg = load_fasta("data/processed/negative_final.fasta")

sequences = seqs_sp1 + seqs_sp2 + seqs_sp4 + seqs_neg
y = np.concatenate([
    np.zeros(len(seqs_sp1)),
    np.ones(len(seqs_sp2)),
    np.full(len(seqs_sp4), 2),
    np.full(len(seqs_neg), 3)
])
print(f"Total: {len(sequences)} sequences, Distribution: {np.bincount(y.astype(int))}")

# --- Load DNABERT-2 ---
print("\nLoading DNABERT-2 encoder...")
wrapper = DNABERTWrapper(device=device)

### Cell 3: Build End-to-End Model + DataLoaders

> [!NOTE]
> **Overfitting Fix**:
> Since our dataset is small (~302 sequences) and DNABERT-2 has 768 channels, using large kernels (e.g. `[3, 5, 7, 9]`) with `128` channels creates ~2.4M parameters, causing immediate overfitting (Train Acc 98% vs Val Acc 33%).
> 
> To fix this, we apply:
> 1. **Smaller kernels `[1, 2, 3]`**: Since DNABERT-2 uses BPE tokens (each token represents ~3-4bp), a kernel of 2 or 3 covers 6-12bp, which is the perfect size for transcription factor binding motifs (GC-box).
> 2. **Fewer channels `branch_channels=32`**: Reduces overall parameter count to ~160K, making the model much more robust.
> 3. **Mathematical Equivalence**: In PyTorch, applying `nn.Conv1d(in_channels=768, out_channels=32, kernel_size=k)` is mathematically identical to Yoon Kim's TextCNN `nn.Conv2d` with kernel size `(k, 768)` tritted along the sequence length, since it computes over all 768 dimensions per step.

In [ ]:
# --- Stratified Split (on raw strings, NOT on embeddings) ---
seq_train, seq_val, y_train, y_val = train_test_split(
    sequences, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {len(seq_train)}, Val: {len(seq_val)}")

# --- On-the-fly tokenizing datasets ---
train_ds = DNAPipelineDataset(seq_train, y_train, wrapper.tokenizer, max_length=105)
val_ds   = DNAPipelineDataset(seq_val,   y_val,   wrapper.tokenizer, max_length=105)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=2)
print(f"Batches: train={len(train_loader)}, val={len(val_loader)}")

# --- Build combined model (Optimized to prevent overfitting) ---
mcnn = MultiScaleCNN(
    embedding_dim=768,
    branch_channels=32,       # Reduced from 128 to 32
    kernel_sizes=[1, 2, 3],   # Reduced from [3, 5, 7, 9]
    num_classes=4,
    dropout_rate=0.6          # Increased dropout
)
model = DNABERT_mCNN(dnabert_model=wrapper.model, mcnn_model=mcnn, freeze_dnabert=True)

# Count trainable params
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}  (mCNN head only)")
print(f"Frozen params:    {total_params - trainable_params:,}  (DNABERT-2 encoder)")

### Cell 4: Train

In [ ]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=15,
    lr=0.001,
    device=device,
    output_dir='models'
)

### Cell 5: Evaluate + Plots

In [ ]:
from IPython.display import Image, display
import os

# Training curves
plot_curves(history, save_dir='figures')
display(Image('figures/mcnn_training_curves.png'))

# Load best checkpoint and evaluate
best_path = os.path.join('models', 'best_mcnn_model.pt')
model.load_state_dict(torch.load(best_path, map_location=device))
print(f"Loaded best model from {best_path}")

class_names = ['SP1', 'SP2', 'SP4', 'Negative']
evaluate_model(model, val_loader, class_names, device=device, save_dir='figures')

print("\n--- Confusion Matrix ---")
display(Image('figures/confusion_matrix.png'))
print("\n--- ROC Curves ---")
display(Image('figures/roc_curves.png'))
print("\n--- Precision-Recall Curves ---")
display(Image('figures/precision_recall_curves.png'))